# Lección 3 · Decisiones sí/no

**Qué vas a aprender**
- **Variables binarias**: decisiones de sí o no (construir o no, abrir o no).
- Por qué **redondear el LP** no sirve.
- **Branch & bound**: cómo el solver encuentra el óptimo sin probar todas las combinaciones.
- **Gap**: cuánto puede faltarle a una solución para ser la mejor.
- Reglas lógicas con binarias: "si A entonces B", "A o B, no los dos".
- **Costo fijo y big-M**: el interruptor de una línea de producción.
- Una cartera de 20 proyectos de energía con los datos en tablas.
- Sensibilidad en un modelo entero: re-solve, no precio sombra.

**Cómo usar este notebook**
- Corre las celdas en orden con `Shift + Enter`.
- Antes de cada celda, adivina qué va a imprimir.

## 0. Preparar el entorno
Igual que en las lecciones 1 y 2.

In [ ]:
%pip install -q pyomo highspy

In [ ]:
import itertools
import io

import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo
from pyomo.opt import TerminationCondition

SOLVER = pyo.SolverFactory("appsi_highs")


def eur(x):
    return f"{x:,.0f}".replace(",", ".") + " €"


def meur(x):
    # millones de euros con coma decimal: 15,3 M€
    return f"{x:.2f}".rstrip("0").rstrip(".").replace(".", ",") + " M€"

## 1. Dónde quedamos
En la lección 2 el plan tenía 43,3 unidades de P01. Para unidades eso está bien: fabricas 43 y listo.

Pero muchas decisiones son de **sí o no**. Construir un parque solar o no. Abrir una línea o no.
"0,83 parques solares" no existe.

**Analogía.** En eToro puedes comprar 0,83 acciones de una empresa: es una decisión continua (LP).
Un departamento en Cracovia no: o lo compras entero o no lo compras. Esa es una **variable binaria**: vale 0 o 1.

Un modelo con variables enteras o binarias es un **MILP** (*mixed-integer linear program*).
Se ve casi igual que un LP. Se resuelve muy distinto.

## 2. Cinco proyectos y un presupuesto
Una empresa de energía tiene **10 M€** para invertir este año. Hay 5 proyectos posibles.
El **VPN** (valor presente neto) es lo que gana la empresa con el proyecto, ya descontado al día de hoy.

| Proyecto | Costo (M€) | VPN (M€) | VPN por M€ invertido |
|---|---|---|---|
| A · parque solar | 5 | 8,0 | 1,60 |
| B · parque eólico | 4 | 6,0 | 1,50 |
| C · subestación | 3 | 4,2 | 1,40 |
| D · baterías | 6 | 9,3 | 1,55 |
| E · digitalización | 2 | 2,6 | 1,30 |

**Las 3 piezas:**
- **Decido:** y_p = 1 si hago el proyecto p, 0 si no.
- **Objetivo:** maximizar Σ VPN_p · y_p.
- **Restricción:** Σ costo_p · y_p ≤ 10.

Se llama **problema de la mochila** (*knapsack*).
**Analogía.** La maleta para viajar a Colombia: 23 kg. Cada cosa pesa y vale algo. O la llevas o no.

In [ ]:
PROY = {  # proyecto: (costo M€, VPN M€)
    "A_solar": (5, 8.0),
    "B_eolico": (4, 6.0),
    "C_subestacion": (3, 4.2),
    "D_baterias": (6, 9.3),
    "E_digital": (2, 2.6),
}
PRESUPUESTO = 10

### 2.1 Fuerza bruta
Cada proyecto es sí o no: 2 × 2 × 2 × 2 × 2 = 2⁵ = **32 combinaciones**. Las probamos todas.

In [ ]:
filas = []
for combo in itertools.product([0, 1], repeat=len(PROY)):
    elegidos = [p for p, y in zip(PROY, combo) if y == 1]
    costo = sum(PROY[p][0] for p in elegidos)
    vpn = sum(PROY[p][1] for p in elegidos)
    filas.append({"elegidos": " + ".join(elegidos) or "(nada)", "costo": costo, "vpn": round(vpn, 2),
                  "factible": costo <= PRESUPUESTO})
todas = pd.DataFrame(filas)

print("Combinaciones:", len(todas), "| factibles:", todas["factible"].sum())
todas[todas["factible"]].nlargest(5, "vpn")

**Lectura:** la mejor es **B + D**: eólico y baterías, 15,3 M€.
Fíjate: no incluye el parque solar, que es el que más VPN da por euro invertido.

**¿Por qué no siempre fuerza bruta?** Con 20 proyectos son 2²⁰ ≈ 1 millón de combinaciones: todavía se puede.
Con 100 proyectos son 2¹⁰⁰ ≈ 10³⁰. Lo mismo que el taller de la lección 1: imposible.

## 3. El atajo tentador: resolver el LP y redondear
Si permitimos que y_p sea cualquier número entre 0 y 1, el modelo es un LP. Se llama la **relajación lineal**
(*LP relaxation*): "relajamos" la regla de que sea 0 o 1.

Armamos el modelo en una función. El parámetro `relajar` elige el tipo de variable:
- `pyo.Binary`: solo 0 o 1.
- `pyo.UnitInterval`: cualquier número entre 0 y 1.

In [ ]:
def armar_mochila(proy=PROY, presupuesto=PRESUPUESTO, relajar=False):
    m = pyo.ConcreteModel()
    m.P = pyo.Set(initialize=list(proy))
    m.y = pyo.Var(m.P, within=pyo.UnitInterval if relajar else pyo.Binary)
    m.obj = pyo.Objective(expr=sum(proy[p][1] * m.y[p] for p in m.P), sense=pyo.maximize)
    m.presupuesto = pyo.Constraint(expr=sum(proy[p][0] * m.y[p] for p in m.P) <= presupuesto)
    return m


def resolver(m):
    SOLVER.solve(m)
    plan = {p: round(pyo.value(m.y[p]), 3) + 0.0 for p in m.P}  # + 0.0 quita el "-0.0"
    print("VPN:", meur(pyo.value(m.obj)), "|", plan)
    return m


lp = resolver(armar_mochila(relajar=True))

**Lectura:** el LP hace todo el solar (y = 1) y 0,83 de las baterías. VPN 15,75 M€.
Llena la mochila por orden de VPN por euro: primero A (1,60), después D (1,55), y D entra a pedazos.

Ahora redondeamos de las dos formas posibles y revisamos.

In [ ]:
def revisar(elegidos, proy=PROY, presupuesto=PRESUPUESTO):
    costo = sum(proy[p][0] for p in elegidos)
    vpn = sum(proy[p][1] for p in elegidos)
    estado = "factible" if costo <= presupuesto else "✗ se pasa del presupuesto"
    print(f"{' + '.join(elegidos) or '(nada)':<22} costo {costo:>2} M€ | VPN {meur(vpn):>9} | {estado}")


y_lp = {p: pyo.value(lp.y[p]) for p in lp.P}
revisar([p for p, v in y_lp.items() if v >= 0.999])  # redondear hacia abajo
revisar([p for p, v in y_lp.items() if v > 0.001])   # redondear hacia arriba
revisar(["B_eolico", "D_baterias"])                    # el óptimo de la fuerza bruta

**Lectura:**
- Hacia abajo: solo el solar. Factible, pero 8,0 M€. Pierdes casi la mitad.
- Hacia arriba: solar + baterías cuesta 11 M€. **No cabe.**
- El óptimo (B + D) ni siquiera incluye al solar, el favorito del LP.

**Por qué pasa.** En el LP sobra presupuesto y se llena con un pedazo. Con piezas enteras importa
**cómo encajan** las piezas, no solo cuánto vale cada una. Es como la maleta: la cosa con mejor valor por kilo
puede dejarte 4 kg vacíos que no llenas con nada.

**Lo que sí sirve del LP: la cota.** El LP puede elegir todo lo que el MILP elige, y más.
Entonces el óptimo entero **nunca** supera 15,75 M€. Esa cota es la base del branch & bound.

## 4. Binarias en Pyomo
Una sola palabra cambia: `within=pyo.Binary`. HiGHS se da cuenta y usa branch & bound.

In [ ]:
mochila = resolver(armar_mochila())

Mismo resultado que la fuerza bruta: **B + D, 15,3 M€**. Pero el solver no probó las 32 combinaciones.
Veamos qué hizo.

## 5. Branch & bound, a mano
El solver repite 3 pasos:
1. **Cota.** Resuelve el LP relajado. Ninguna solución entera de esa rama vale más que eso.
2. **Ramificar** (*branch*). Si una variable sale fraccional (D = 0,83), parte el problema en dos:
   una rama con D = 0 y otra con D = 1. Las dos juntas cubren todo.
3. **Podar** (*bound*). Una rama se descarta si:
   - es infactible;
   - su cota no supera la mejor solución entera que ya tengo;
   - su LP da una solución entera (no hay nada más que buscar ahí; si es mejor, se guarda).

**Analogía.** Buscas departamento en Cracovia barrio por barrio. Ya viste uno en Podgórze que te gusta.
Si el mejor departamento imaginable de Nowa Huta (su cota) es peor que ese, no vas a visitar Nowa Huta.

Abajo está el algoritmo completo en 30 líneas. Cada fila de la tabla es un **nodo** del árbol.

In [ ]:
def cota_lp(proy, presupuesto, fijas):
    """Resuelve el LP relajado con algunas variables fijas. Devuelve (cota, plan) o (None, None) si es infactible."""
    m = armar_mochila(proy, presupuesto, relajar=True)
    for p, valor in fijas.items():
        m.y[p].fix(valor)
    res = SOLVER.solve(m, load_solutions=False)
    if res.solver.termination_condition != TerminationCondition.optimal:
        return None, None
    m.solutions.load_from(res)
    return pyo.value(m.obj), {p: round(pyo.value(m.y[p]), 3) for p in m.P}


def branch_and_bound(proy=PROY, presupuesto=PRESUPUESTO):
    pendientes = [({}, float("inf"))]  # (variables fijas, cota del nodo padre)
    mejor_vpn, mejor_plan, log = 0.0, None, []
    while pendientes:
        fijas, _ = pendientes.pop()  # el último que entró: búsqueda en profundidad
        nodo = " , ".join(f"{p[0]}={v}" for p, v in fijas.items()) or "raíz"
        cota, plan = cota_lp(proy, presupuesto, fijas)
        if cota is None:
            accion = "infactible → podar"
        elif cota <= mejor_vpn + 1e-9:
            accion = "cota ≤ mejor → podar"
        else:
            fraccionales = [p for p, v in plan.items() if 1e-6 < v < 1 - 1e-6]
            if not fraccionales:
                mejor_vpn, mejor_plan = cota, [p for p, v in plan.items() if v > 0.5]
                accion = "entera → nueva mejor"
            else:
                p = fraccionales[0]
                accion = f"{p}={plan[p]} → ramificar"
                pendientes.append(({**fijas, p: 0}, cota))
                pendientes.append(({**fijas, p: 1}, cota))
        # La mejor cota global: lo máximo que todavía podría valer alguna rama sin revisar
        cota_global = max([c for _, c in pendientes] + [mejor_vpn])
        gap = (cota_global - mejor_vpn) / mejor_vpn if mejor_vpn > 0 else float("inf")
        log.append({"nodo": nodo, "cota_lp": cota, "accion": accion, "mejor": mejor_vpn,
                    "cota_global": cota_global, "gap_%": round(100 * gap, 2)})
    print("Mejor plan:", mejor_plan, "→", meur(mejor_vpn), "| nodos:", len(log), "de", 2 ** len(proy), "combinaciones")
    return pd.DataFrame(log)


branch_and_bound()

**Cómo se lee el árbol** (las letras son la inicial de cada proyecto):
1. **Raíz:** cota 15,75. D sale 0,83 → dos ramas.
2. **D = 1:** cota 15,70. Ahora A sale 0,8 → dos ramas.
3. **D = 1, A = 1:** cuesta 11 M€. Infactible → podar.
4. **D = 1, A = 0:** el LP da B + D, todo entero. **Primera solución: 15,3.**
5. **D = 0:** cota 15,4. Todavía puede ganarle a 15,3 → ramificar por C.
6. y 7. **D = 0, C = 1** (cota 15,2) y **D = 0, C = 0** (cota 15,3): ninguna supera 15,3 → podar.

**7 nodos en vez de 32.** Con 100 proyectos la diferencia es de millones de veces.
La poda funciona mejor cuando la **cota es ajustada** (cercana al óptimo entero). Recuerda esto para la sección 7.

### El gap
Mira las últimas columnas (`inf` = todavía no hay ningún plan). En todo momento el solver conoce dos números:
- **Mejor solución** encontrada (*incumbent*): 15,3. Es un plan real.
- **Cota global** (*best bound*): lo máximo que podría valer una rama sin revisar.

El **gap** es la distancia entre los dos: (cota − mejor) / mejor.
Después del nodo 4 el gap es 2,9 %: "tengo un plan de 15,3 y seguro que ninguno pasa de 15,75".
Cuando el gap llega a 0, está **demostrado** que el plan es el óptimo.

Es la misma idea de la dualidad débil de la lección 2: una cota por arriba y una solución por abajo que se tocan.

## 6. Reglas lógicas con binarias
Con binarias puedes escribir reglas del negocio como restricciones lineales.

| Regla | Restricción | Ejemplo |
|---|---|---|
| Si A, entonces B | y_A ≤ y_B | baterías solo si hay parque solar |
| A o B, no los dos | y_A + y_B ≤ 1 | dos diseños para el mismo terreno |
| Al menos k de un grupo | Σ y ≥ k | el regulador pide 2 proyectos de red |
| Si A y B, entonces C | y_C ≥ y_A + y_B − 1 | dos parques juntos exigen una subestación |

Prueba la primera fila con la tabla de verdad: si y_A = 1, la restricción obliga y_B = 1.
Si y_A = 0, y_B puede ser 0 o 1. Es exactamente "si A, entonces B".

**Ejemplo.** Las baterías se instalan junto al parque solar: sin solar no hay baterías.

In [ ]:
m = armar_mochila()
m.baterias_con_solar = pyo.Constraint(expr=m.y["D_baterias"] <= m.y["A_solar"])
resolver(m);

**Lectura:** solar + baterías no cabe (11 M€), y baterías solas ya no se permiten.
El plan cambia a **A + C + E** (solar, subestación, digitalización): 14,8 M€.
La regla le cuesta 0,5 M€ a la empresa. Ese número sirve para discutir la regla con el área técnica.

## 7. Costo fijo y big-M: el interruptor
Vuelve el taller de la lección 1: válvulas (300 €) y conectores (200 €). El plan era 20 válvulas y 60 conectores: 18.000 €.

Ahora cada línea tiene un **costo fijo** semanal por estar encendida, se fabrique 1 unidad o 80:
- línea de válvulas: 2.500 €
- línea de conectores: 3.000 €

**Analogía.** El gimnasio. Pagas la cuota del mes (costo fijo) si vas una vez o veinte.
Si no vas nunca, no te inscribes y no pagas.

Necesitamos una binaria por línea (y = 1 si está encendida) y una regla que las conecte con la producción:

> **V ≤ M · y_V**

- Si y_V = 0, entonces V ≤ 0: sin línea no hay válvulas.
- Si y_V = 1, entonces V ≤ M: la restricción no estorba si M es suficientemente grande.

M es el famoso **big-M**. La pregunta es qué tan grande.

In [ ]:
FIJO = {"valvula": 2500, "conector": 3000}


def taller_fijo(M_V=40, M_C=80, fijo=FIJO, relajar=False):
    m = pyo.ConcreteModel()
    m.V = pyo.Var(within=pyo.NonNegativeReals)
    m.C = pyo.Var(within=pyo.NonNegativeReals)
    tipo = pyo.UnitInterval if relajar else pyo.Binary
    m.yV = pyo.Var(within=tipo)  # 1 = línea de válvulas encendida
    m.yC = pyo.Var(within=tipo)  # 1 = línea de conectores encendida
    m.obj = pyo.Objective(expr=300 * m.V + 200 * m.C - fijo["valvula"] * m.yV - fijo["conector"] * m.yC,
                          sense=pyo.maximize)
    m.mec = pyo.Constraint(expr=2 * m.V + m.C <= 100)
    m.pru = pyo.Constraint(expr=m.V + m.C <= 80)
    m.mer = pyo.Constraint(expr=m.V <= 40)
    m.switch_V = pyo.Constraint(expr=m.V <= M_V * m.yV)  # el interruptor
    m.switch_C = pyo.Constraint(expr=m.C <= M_C * m.yC)
    SOLVER.solve(m)
    return m


m = taller_fijo()
print(f"V={pyo.value(m.V):.0f}, C={pyo.value(m.C):.0f}, líneas: válvulas={pyo.value(m.yV):.0f}, conectores={pyo.value(m.yC):.0f}"
      f" → {eur(pyo.value(m.obj))}")

**Lectura:** se **apaga la línea de válvulas**. Solo 80 conectores: 16.000 − 3.000 = 13.000 €.

Comprobación a mano:
- Las dos líneas: 18.000 − 2.500 − 3.000 = 12.500 €.
- Solo conectores: 80 × 200 − 3.000 = **13.000 €**.
- Solo válvulas: 40 × 300 − 2.500 = 9.500 €.

Las 20 válvulas del plan de la lección 1 dejan 2.000 € de ganancia extra, y su línea cuesta 2.500 €. No se pagan.

### ¿De dónde salió M = 40 y M = 80?
De las otras restricciones. Nunca habrá más de 40 válvulas (mercado) ni más de 80 conectores (prueba).
Es el M **más chico que sigue siendo seguro**. Comparemos con un M exagerado y con uno muy chico.

In [ ]:
casos = {"M justo (40, 80)": dict(M_V=40, M_C=80),
         "M enorme (1.000.000)": dict(M_V=1e6, M_C=1e6),
         "M muy chico (C ≤ 50)": dict(M_V=40, M_C=50)}
filas = []
for nombre, M in casos.items():
    entero = taller_fijo(**M)
    relajado = taller_fijo(**M, relajar=True)
    filas.append({"caso": nombre, "MILP": pyo.value(entero.obj), "cota LP": round(pyo.value(relajado.obj), 1),
                  "V": pyo.value(entero.V) + 0.0, "C": pyo.value(entero.C) + 0.0,
                  "gap en la raíz %": round(100 * (pyo.value(relajado.obj) / pyo.value(entero.obj) - 1), 1)})
pd.DataFrame(filas)

**Lectura:**
- **M justo:** óptimo 13.000 €. La cota LP es 14.500 €: gap inicial 11,5 %.
- **M enorme:** el mismo óptimo, pero la cota LP es casi 18.000 €: gap inicial 38 %.
  En el LP relajado, y_V = 20 / 1.000.000 basta para "encender" la línea. El costo fijo casi desaparece.
  Es como pagar 0,002 % de la cuota del gimnasio y entrenar todo el mes.
  La cota es mala → el solver poda poco → con modelos grandes tarda horas en vez de segundos.
- **M muy chico:** la respuesta **está mal** (12.000 €). C ≤ 50 corta planes que sí eran posibles.

**Regla de trabajo:** M = el valor más chico que estás seguro de que nunca se va a superar.
Calcúlalo desde los datos (capacidades, demanda), no pongas 10⁶ "por si acaso".
Además, un M enorme trae problemas numéricos: el solver puede aceptar y = 0,0000001 como si fuera 0.

## 8. El modelo crece: cartera de 20 proyectos
Una empresa de energía en Polonia arma su cartera de inversión para 2027 y 2028.
Misma estrategia de la lección 2: los datos viven en **tablas**, el modelo se arma desde las tablas,
y la solución vuelve a tablas.

Cuatro tablas, como en SQL:
1. **Proyectos:** tipo, inversión (*capex*) de cada año, VPN y megavatios (MW) renovables.
2. **Dependencias:** un proyecto requiere otro (y_p ≤ y_requiere).
3. **Exclusiones:** dos proyectos que no pueden ir juntos (y_a + y_b ≤ 1).
4. **Presupuesto** por año.

Y una regla del regulador: al menos 2 proyectos de red.

In [ ]:
df_proy = pd.read_csv(io.StringIO("""proyecto,tipo,capex_2027,capex_2028,vpn,mw
solar_sur,solar,12,4,18.5,60
solar_techos,solar,4,2,5.1,15
solar_norte,solar,10,6,15.8,50
baterias_sur,baterias,6,5,10.2,0
baterias_norte,baterias,5,6,8.4,0
eolico_costa_a,eolico,14,10,24.0,80
eolico_costa_b,eolico,11,9,19.5,65
eolico_monte,eolico,9,8,13.2,45
repotenciar_eolico,eolico,5,3,7.9,20
subestacion_norte,red,7,4,6.5,0
subestacion_sur,red,6,3,5.8,0
linea_costa,red,8,7,7.0,0
medidores_inteligentes,red,3,4,4.4,0
scada_nuevo,digital,4,2,3.9,0
gemelo_digital,digital,3,3,6.1,0
mantenimiento_predictivo,digital,2,2,4.8,0
biogas_granjas,biogas,6,4,8.0,12
calor_residual,eficiencia,3,2,4.5,0
bombas_calor,eficiencia,4,3,5.6,0
hidrogeno_piloto,hidrogeno,9,8,9.5,5
"""))

df_dep = pd.DataFrame([("baterias_sur", "solar_sur"), ("baterias_norte", "solar_norte"),
                       ("solar_norte", "subestacion_norte"), ("eolico_costa_a", "linea_costa"),
                       ("eolico_costa_b", "linea_costa"), ("gemelo_digital", "scada_nuevo"),
                       ("mantenimiento_predictivo", "scada_nuevo")], columns=["proyecto", "requiere"])

df_exc = pd.DataFrame([("eolico_costa_a", "eolico_costa_b"),   # mismo permiso en el mar: uno de los dos diseños
                       ("biogas_granjas", "hidrogeno_piloto")],  # mismo equipo de ingeniería
                      columns=["a", "b"])

PRESUP = {"2027": 55, "2028": 40}

print("Si hiciéramos todo:", df_proy[["capex_2027", "capex_2028", "vpn"]].sum().to_dict())
df_proy.head()

Hacer todo cuesta 131 M€ en 2027 y 95 M€ en 2028. Solo hay 55 y 40. Hay que elegir.

### 8.1 El modelo
Las mismas piezas de la mochila, más una restricción por cada fila de las tablas de reglas.
`mip_gap` es la tolerancia de gap: lo usamos en la sección 9.

In [ ]:
def cartera(df_proy=df_proy, df_dep=df_dep, df_exc=df_exc, presup=PRESUP, min_red=2, min_mw=0,
            relajar=False, mip_gap=None, log=False):
    d = df_proy.set_index("proyecto")
    m = pyo.ConcreteModel()
    m.P = pyo.Set(initialize=list(d.index))
    m.y = pyo.Var(m.P, within=pyo.UnitInterval if relajar else pyo.Binary)
    m.obj = pyo.Objective(expr=sum(d.loc[p, "vpn"] * m.y[p] for p in m.P), sense=pyo.maximize)
    m.presup = pyo.Constraint(list(presup), rule=lambda m, a: sum(d.loc[p, f"capex_{a}"] * m.y[p] for p in m.P) <= presup[a])
    m.dep = pyo.Constraint(df_dep.index, rule=lambda m, i: m.y[df_dep.proyecto[i]] <= m.y[df_dep.requiere[i]])
    m.exc = pyo.Constraint(df_exc.index, rule=lambda m, i: m.y[df_exc.a[i]] + m.y[df_exc.b[i]] <= 1)
    m.red = pyo.Constraint(expr=sum(m.y[p] for p in m.P if d.loc[p, "tipo"] == "red") >= min_red)
    m.mw = pyo.Constraint(expr=sum(d.loc[p, "mw"] * m.y[p] for p in m.P) >= min_mw)
    solver = pyo.SolverFactory("appsi_highs")
    if mip_gap is not None:
        solver.config.mip_gap = mip_gap
    if log:  # el log de HiGHS va a un archivo para poder leerlo en el notebook
        solver.highs_options = {"log_file": "highs.log", "log_to_console": False}
    res = solver.solve(m)
    m.cota = res.problem.upper_bound  # best bound (maximizamos: la cota es por arriba)
    return m


def elegidos(m, umbral=0.5):
    return [p for p in m.P if pyo.value(m.y[p]) > umbral]


mc = cartera()
print("VPN:", meur(pyo.value(mc.obj)), "| proyectos:", len(elegidos(mc)))
print(elegidos(mc))

### 8.2 ¿Y si redondeamos el LP?
Hagamos la prueba de la sección 3 con el modelo grande. `revisar_cartera` mide cada plan contra **todas** las reglas.

In [ ]:
def revisar_cartera(sel, nombre, presup=PRESUP, min_red=2):
    d = df_proy.set_index("proyecto")
    fila = {"plan": nombre, "proyectos": len(sel), "vpn": round(d.loc[sel, "vpn"].sum(), 2)}
    for a in presup:
        fila[f"capex_{a}"] = d.loc[sel, f"capex_{a}"].sum()
    fila["dependencias_ok"] = all(r.proyecto not in sel or r.requiere in sel for r in df_dep.itertuples())
    fila["exclusiones_ok"] = all(not (r.a in sel and r.b in sel) for r in df_exc.itertuples())
    fila["proyectos_red"] = int((d.loc[sel, "tipo"] == "red").sum())
    fila["factible"] = (all(fila[f"capex_{a}"] <= presup[a] for a in presup) and fila["dependencias_ok"]
                        and fila["exclusiones_ok"] and fila["proyectos_red"] >= min_red)
    return fila


lp20 = cartera(relajar=True)
y20 = {p: pyo.value(lp20.y[p]) for p in lp20.P}
print("LP relajado:", meur(pyo.value(lp20.obj)), "| variables fraccionales:", sum(0.001 < v < 0.999 for v in y20.values()))

pd.DataFrame([
    revisar_cartera([p for p, v in y20.items() if v >= 0.999], "LP redondeado hacia abajo"),
    revisar_cartera([p for p, v in y20.items() if v >= 0.5], "LP redondeado al más cercano"),
    revisar_cartera(elegidos(mc), "MILP"),
])

**Lectura:**
- El LP vale 84,73 M€ con 7 variables fraccionales. Por ejemplo, eólico costa A = 0,5 y costa B = 0,5: medio parque de cada diseño.
- **Hacia abajo:** cabe en el presupuesto, pero queda 1 solo proyecto de red. **Rompe la regla del regulador.** Y vale 55,8.
- **Al más cercano:** 106,3 M€ en el papel. Se pasa 13 M€ en 2027 y hace los dos diseños de costa. **Imposible.**
- **MILP:** 83,4 M€ y cumple todo.

Las dos formas de redondear dan planes **infactibles**. Con restricciones de "al menos" y de exclusión,
redondear casi nunca funciona. Esta es la respuesta a "¿por qué no redondear?".

Lo que sí da el LP: **ninguna cartera vale más de 84,73 M€**. El MILP está a 1,6 % de esa cota.

### 8.3 La solución, de vuelta a tablas

In [ ]:
res = df_proy.copy()
res["elegido"] = res["proyecto"].map(lambda p: round(pyo.value(mc.y[p])) == 1)
res["vpn_por_meur"] = (res["vpn"] / (res["capex_2027"] + res["capex_2028"])).round(2)

uso = pd.DataFrame([{"año": a, "usado": res.loc[res.elegido, f"capex_{a}"].sum(), "presupuesto": PRESUP[a]} for a in PRESUP])
uso["holgura"] = uso["presupuesto"] - uso["usado"]
uso

**¿Dónde está el valor?** Los 5 proyectos elegidos con más VPN.

In [ ]:
res[res.elegido].nlargest(5, "vpn")[["proyecto", "tipo", "capex_2027", "capex_2028", "vpn"]]

**¿Qué buenos proyectos quedaron fuera?** Los no elegidos con más VPN por M€ invertido.

In [ ]:
res[~res.elegido].nlargest(5, "vpn_por_meur")[["proyecto", "tipo", "capex_2027", "capex_2028", "vpn", "vpn_por_meur"]]

**Lectura para el comité de inversión:**
- 2027 usa los 55 M€: es el año que manda. En 2028 sobra 1 M€.
- El eólico costa A (24 M€ de VPN) entra junto con la línea de costa, que es su requisito.
- **solar_norte** y **repotenciar_eolico** dejan 0,99 € de VPN por euro y quedan fuera.
  La **linea_costa** deja solo 0,47 y entra: sola no vale, pero sin ella no hay eólico costa A.
- Ordenar por "VPN por euro" no es el plan. Lo que importa es cómo encajan las piezas, igual que en la sección 3.

## 9. El gap en el solver real
Corremos la cartera con `log=True`: HiGHS escribe su árbol de branch & bound en `highs.log`. Imprimimos la tabla.
Busca las columnas **BestBound** (cota global), **BestSol** (mejor solución) y **Gap**.

In [ ]:
cartera(log=True)
texto = open("highs.log").read()
print(texto[texto.rfind("Src  Proc."):texto.rfind("Solving report")])

**Cómo se lee** (los números exactos pueden cambiar con la versión de HiGHS):
- Cada fila es un momento de la búsqueda. La letra de la izquierda dice qué heurística encontró la solución.
- **BestSol** sube: 28,7 → 62,8 → 67,3 → 82,3 → 83,4. Planes cada vez mejores.
- **BestBound** baja: 161 → 83,8 → 83,4. La cota se ajusta con cortes y ramificaciones.
- Cuando se tocan, **Gap = 0 %** y el solver para: está demostrado que 83,4 es el óptimo.

En modelos grandes esto puede tardar horas. Por eso se le dice al solver cuándo parar:
- `mip_gap = 0.01`: para cuando el gap baje de 1 %.
- `time_limit = 60`: para a los 60 segundos (`solver.config.time_limit`).

Veamos qué entrega con distintas tolerancias.

In [ ]:
filas = []
for tolerancia in [0.0001, 0.02, 0.30]:
    mg = cartera(mip_gap=tolerancia)
    filas.append({"mip_gap": f"{tolerancia:.2%}", "vpn_entregado": pyo.value(mg.obj), "cota": round(mg.cota, 2),
                  "gap_real_%": round(100 * (mg.cota - pyo.value(mg.obj)) / pyo.value(mg.obj), 2)})
pd.DataFrame(filas)

**Lectura:**
- Con 0,01 % entrega el óptimo, 83,4.
- Con 2 % para en 82,3: su gap ya era 1,8 %. Deja 1,1 M€ en la mesa.
- Con 30 % para en la primera solución decente que encuentra.

**Para el jefe**, siempre los dos números: *"Esta cartera vale 82,3 M€ y ninguna cartera posible vale más de 83,8 M€."*
Así el jefe decide si vale la pena esperar más.

## 10. Sensibilidad en un MILP
En la lección 2 el precio sombra te decía cuánto vale una unidad más de un recurso.
En un MILP la ganancia cambia **a saltos**, y el precio sombra deja de tener sentido. Dos salidas:

### 10.1 Fijar las binarias y leer el LP que queda
Funciona cuando hay variables continuas, como en el taller con costo fijo.
Fijamos la decisión sí/no (línea de válvulas apagada, conectores encendida) y lo que queda es un LP.

In [ ]:
m = taller_fijo()
for y, x, switch in [(m.yV, m.V, m.switch_V), (m.yC, m.C, m.switch_C)]:
    decision = round(pyo.value(y))
    y.fix(decision)
    switch.deactivate()  # el interruptor ya no hace falta: la decisión está tomada
    if decision == 0:
        x.fix(0)  # línea apagada: no se produce
pyo.TransformationFactory("core.relax_integer_vars").apply_to(m)  # quita la etiqueta "entera": ahora es un LP
m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
SOLVER.solve(m)

print("Ganancia:", eur(pyo.value(m.obj)))
pd.DataFrame([{"restriccion": r.name, "usado": pyo.value(r.body), "limite": pyo.value(r.upper), "precio_sombra": m.dual[r]}
              for r in [m.mec, m.pru, m.mer]])

**Lectura:** con la línea de válvulas apagada, una hora más de prueba vale **200 €**: un conector más.
Vale **mientras la decisión sí/no no cambie**. Si con muchas horas extra conviene encender las válvulas,
el número deja de valer. Hay que re-resolver el MILP.

### 10.2 Re-resolver por escenarios
En la cartera todas las variables son binarias. Fijarlas no deja nada continuo.
La única forma honesta: resolver el MILP para cada presupuesto y mirar la curva.

In [ ]:
barrido = pd.DataFrame([{"presupuesto_2027": b, "vpn": pyo.value(cartera(presup={**PRESUP, "2027": b}).obj)}
                        for b in range(45, 71)])
barrido["vale_1_meur_mas"] = barrido["vpn"].diff().shift(-1).round(2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.step(barrido["presupuesto_2027"], barrido["vpn"], where="post")
ax.axvline(55, color="gray", linestyle="--", label="hoy: 55 M€")
ax.set(xlabel="Presupuesto 2027 (M€)", ylabel="VPN de la cartera (M€)", title="VPN según presupuesto 2027")
ax.legend()
plt.show()

barrido[(barrido.presupuesto_2027 >= 53) & (barrido.presupuesto_2027 <= 60)]

**Lectura:**
- La curva es una **escalera irregular**, no los tramos rectos de la lección 2.
- 1 M€ más en 2027 vale 1,1 M€ (de 55 a 56). Pero de 57 a 58 vale **0**, y de 56 a 57 vale 2,3.
- No hay un número único que llevar al jefe. Se lleva la tabla por escenarios.

**Regla de trabajo:** en un MILP, la sensibilidad es **re-solve por escenarios**.
Con 20 proyectos son 26 corridas de milisegundos. Con modelos grandes, se eligen pocos escenarios con el negocio.

## 11. Resumen: conceptos de hoy
| Concepto | Qué es | En la lección | En inglés |
|---|---|---|---|
| Variable binaria | Decisión que vale 0 o 1 | hacer o no un proyecto | binary variable |
| MILP | LP con algunas variables enteras | cartera de 20 proyectos | mixed-integer linear program |
| Relajación lineal | El MILP con las binarias entre 0 y 1 | 15,75 M€, baterías = 0,83 | LP relaxation |
| Branch & bound | Partir en ramas, acotar con el LP, podar | 7 nodos en vez de 32 | branch and bound |
| Mejor solución | El mejor plan entero encontrado | 15,3 M€ | incumbent |
| Cota | Lo máximo que puede valer cualquier plan | 15,75 M€ | best bound, dual bound |
| Gap | (cota − mejor) / mejor | 2 % → para en 82,3 | MIP gap, optimality gap |
| Big-M | x ≤ M·y: sin interruptor no hay producción | M = 40 válvulas, 80 conectores | big-M constraint |
| Costo fijo | Se paga si y = 1, sin importar cuánto produzcas | 2.500 € la línea de válvulas | fixed charge |
| Regla lógica | "Si A, entonces B" como y_A ≤ y_B | baterías solo con solar | logical constraint |

## 12. Ejercicios
1. **¿Cuándo vuelven las válvulas?** En el taller, sube el costo fijo de la línea de conectores de 3.000 € a 8.000 €,
   de 500 en 500. ¿A partir de qué valor conviene apagar los conectores y fabricar solo válvulas?
   Primero calcúlalo a mano con la comprobación de la sección 7.
2. **Regla "si A y B, entonces C".** El operador de red avisa: si se hacen `solar_sur` y `eolico_costa_a` a la vez,
   hay que hacer también `subestacion_sur`. Agrega la restricción a la cartera y resuelve. ¿Cuánto VPN cuesta la regla?
   Pista: `m = cartera()` ya viene resuelto; arma la restricción y vuelve a llamar `SOLVER.solve(m)`.
3. **Megavatios renovables.** El gobierno exige al menos 150 MW renovables en la cartera.
   ¿Cuántos MW tiene hoy el plan? ¿Cuánto VPN cuesta la exigencia? ¿Qué proyectos cambian?

In [ ]:
# Escribe aquí tus intentos

## En la entrevista
Pregunta típica: *"Why can't you just solve the LP relaxation and round?"*
> *"Rounding down is often far from optimal, rounding up is often infeasible, and with logical constraints
> (dependencies, at-least-k, exclusions) both usually break something. In my portfolio example both roundings were infeasible.
> What the LP relaxation does give is a bound. Branch and bound uses that bound to prune branches,
> and the gap between the best solution and the best bound tells me how far from optimal I can be."*

Seguimiento: *"How do you choose big-M?"*
> *"As the smallest value that is guaranteed to be valid, derived from the data: capacity, demand or another constraint.
> A huge M weakens the LP relaxation, so the solver prunes less, and it causes numerical trouble.
> A too-small M cuts off feasible solutions and silently gives a wrong answer."*

## Soluciones (no mires antes de intentar)

In [ ]:
# Ejercicio 1
for fijo_c in range(3000, 8001, 500):
    mt = taller_fijo(fijo={"valvula": 2500, "conector": fijo_c})
    print(f"costo fijo conectores {eur(fijo_c):>8} → V={pyo.value(mt.V) + 0.0:>4.0f}, C={pyo.value(mt.C) + 0.0:>4.0f}, ganancia {eur(pyo.value(mt.obj))}")

Solo conectores gana 16.000 − F. Solo válvulas gana 9.500. Empatan en F = 6.500 €.
Por encima de 6.500 € se apaga la línea de conectores y se fabrican 40 válvulas.
Las dos líneas juntas nunca convienen: siempre ganan 500 € menos que solo conectores.

In [ ]:
# Ejercicio 2
m = cartera()
base = pyo.value(m.obj)
m.operador = pyo.Constraint(expr=m.y["subestacion_sur"] >= m.y["solar_sur"] + m.y["eolico_costa_a"] - 1)
SOLVER.solve(m)
print("VPN con la regla:", meur(pyo.value(m.obj)), "| cuesta", meur(base - pyo.value(m.obj)))
print("Entran:", sorted(set(elegidos(m)) - set(elegidos(mc))))
print("Salen: ", sorted(set(elegidos(mc)) - set(elegidos(m))))

Si y_solar = 1 y y_eolico = 1, la restricción queda y_sub ≥ 1: obliga la subestación.
Si cualquiera de los dos es 0, queda y_sub ≥ 0: no obliga nada.

La regla cuesta solo 0,3 M€. El solver no paga la subestación sur: saca el eólico costa A con su línea
y mete el solar norte, sus baterías y la subestación norte. Una regla pequeña puede cambiar media cartera.

In [ ]:
# Ejercicio 3
mw = df_proy.set_index("proyecto")["mw"]
print("MW del plan actual:", mw[elegidos(mc)].sum())
m150 = cartera(min_mw=150)
print("VPN con ≥ 150 MW:", meur(pyo.value(m150.obj)), "| cuesta", meur(pyo.value(mc.obj) - pyo.value(m150.obj)),
      "| MW:", mw[elegidos(m150)].sum())
print("Entran:", sorted(set(elegidos(m150)) - set(elegidos(mc))))
print("Salen: ", sorted(set(elegidos(mc)) - set(elegidos(m150))))

El plan tiene 140 MW (solar sur 60 + eólico costa A 80). Con la exigencia llega a 160 MW:
entra la repotenciación eólica (20 MW) y bombas de calor; salen baterías sur y calor residual, para hacer espacio en 2027.
Cuesta 1,2 M€ de VPN.
Ese es el número que la empresa lleva a la discusión con el gobierno: el costo de la regla.